In [ ]:
# Import necessary libraries
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import skew, boxcox_normmax
from scipy.special import boxcox1p

from sklearn.model_selection import KFold, cross_val_score
from sklearn.preprocessing import RobustScaler
from sklearn.linear_model import Lasso, Ridge
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.neural_network import MLPRegressor
from xgboost import XGBRegressor
from lightgbm import LGBMRegressor

# Suppress warnings for a cleaner notebook
import warnings
warnings.filterwarnings('ignore')

# Load the datasets
train = pd.read_csv('/kaggle/input/house-prices-advanced-regression-techniques/train.csv')
test = pd.read_csv('/kaggle/input/house-prices-advanced-regression-techniques/test.csv')

# Keep the IDs for submission later
train_ID = train['Id']
test_ID = test['Id']
train.drop('Id', axis=1, inplace=True)
test.drop('Id', axis=1, inplace=True)

print(f"Train set shape: {train.shape}")
print(f"Test set shape: {test.shape}")

In [ ]:
# Plot the distribution of the target variable, SalePrice
sns.displot(train['SalePrice'], kde=True)
plt.title('Distribution of SalePrice')
plt.show()

# The distribution is right-skewed. A log transformation can help normalize it.
train['SalePrice'] = np.log1p(train['SalePrice'])

# Plot the transformed SalePrice
sns.displot(train['SalePrice'], kde=True)
plt.title('Distribution of Log-Transformed SalePrice')
plt.show()

In [ ]:
# Combine train and test data for easier preprocessing
all_data = pd.concat((train.loc[:,'MSSubClass':'SaleCondition'],
                      test.loc[:,'MSSubClass':'SaleCondition']))

# Function to display missing value percentages
def display_missing(df):
    missing = df.isnull().sum()
    missing = missing[missing > 0]
    missing.sort_values(inplace=True, ascending=False)
    missing_percent = (missing / len(df)) * 100
    missing_df = pd.concat([missing, missing_percent], axis=1, keys=['Total', 'Percent'])
    print(missing_df)

display_missing(all_data)

# Impute missing values based on data_description.txt
# For these features, 'NA' means the absence of the feature.
for col in ('PoolQC', 'MiscFeature', 'Alley', 'Fence', 'FireplaceQu', 'GarageType', 'GarageFinish', 'GarageQual', 'GarageCond', 'BsmtQual', 'BsmtCond', 'BsmtExposure', 'BsmtFinType1', 'BsmtFinType2', 'MasVnrType'):
    all_data[col] = all_data[col].fillna('None')

# For numerical features, 'NA' likely means 0.
for col in ('GarageYrBlt', 'GarageArea', 'GarageCars', 'BsmtFinSF1', 'BsmtFinSF2', 'BsmtUnfSF','TotalBsmtSF', 'BsmtFullBath', 'BsmtHalfBath', 'MasVnrArea'):
    all_data[col] = all_data[col].fillna(0)

# 'LotFrontage': Impute with the median LotFrontage of the neighborhood.
all_data['LotFrontage'] = all_data.groupby('Neighborhood')['LotFrontage'].transform(
    lambda x: x.fillna(x.median()))

# Fill remaining categorical NaNs with the mode
all_data['MSZoning'] = all_data['MSZoning'].fillna(all_data['MSZoning'].mode()[0])
all_data['Electrical'] = all_data['Electrical'].fillna(all_data['Electrical'].mode()[0])
all_data['KitchenQual'] = all_data['KitchenQual'].fillna(all_data['KitchenQual'].mode()[0])
all_data['Exterior1st'] = all_data['Exterior1st'].fillna(all_data['Exterior1st'].mode()[0])
all_data['Exterior2nd'] = all_data['Exterior2nd'].fillna(all_data['Exterior2nd'].mode()[0])
all_data['SaleType'] = all_data['SaleType'].fillna(all_data['SaleType'].mode()[0])
all_data['Functional'] = all_data['Functional'].fillna('Typ') # Per data description

# Check if any missing values remain
print("\nRemaining missing values after imputation:")
display_missing(all_data)

In [ ]:
# Some numerical features are actually categorical
all_data['MSSubClass'] = all_data['MSSubClass'].apply(str)
all_data['OverallCond'] = all_data['OverallCond'].astype(str)
all_data['YrSold'] = all_data['YrSold'].astype(str)
all_data['MoSold'] = all_data['MoSold'].astype(str)

# Create a new feature for total square footage
all_data['TotalSF'] = all_data['TotalBsmtSF'] + all_data['1stFlrSF'] + all_data['2ndFlrSF']

# Identify skewed numerical features
numeric_feats = all_data.dtypes[all_data.dtypes != "object"].index
skewed_feats = all_data[numeric_feats].apply(lambda x: skew(x.dropna()))
skewed_feats = skewed_feats[skewed_feats > 0.75]
skewed_feats = skewed_feats.index

# Apply Box-Cox transformation to skewed features
for feat in skewed_feats:
    # A more robust check: transform only if there is more than one unique value
    if all_data[feat].nunique() > 1:
        s = all_data[feat].replace([np.inf, -np.inf], np.nan)

        # Ensure inputs to boxcox1p are strictly > -1 (since boxcox1p(x) = boxcox(1+x))
        finite_vals = s.dropna().values
        if finite_vals.size == 0:
            continue  # nothing to fit/transform

        min_val = np.min(finite_vals)
        shift = 0.0
        if min_val <= -1:
            shift = (-1.0 - min_val) + 1e-6  # tiny epsilon to be strictly > -1

        # Compute lambda on valid, shifted data
        try:
            lam = boxcox_normmax((finite_vals + shift) + 1)
        except Exception:
            # If optimizer still struggles, skip this feature (keeps functionality stable)
            continue

        # Transform, preserving NaNs
        s_trans = s.copy()
        mask = s_trans.notna()
        s_trans.loc[mask] = boxcox1p(s_trans.loc[mask] + shift, lam)

        all_data[feat] = s_trans

# Convert categorical variables into dummy/indicator variables (one-hot encoding)
all_data = pd.get_dummies(all_data)

# Separate back into training and testing sets
X = all_data[:train.shape[0]]
X_test = all_data[train.shape[0]:]
y = train.SalePrice

print(f"Training features shape: {X.shape}")
print(f"Test features shape: {X_test.shape}")

In [ ]:
# Define cross-validation strategy
kf = KFold(n_splits=5, shuffle=True, random_state=42)

# Function to calculate cross-validated RMSE
def cv_rmse(model, X=X):
    rmse = np.sqrt(-cross_val_score(model, X, y, scoring="neg_mean_squared_error", cv=kf))
    return rmse

# Initialize models to test
models = {
    "Lasso": Lasso(alpha=0.0005, random_state=42),
    "Ridge": Ridge(alpha=10, random_state=42),
    "RandomForest": RandomForestRegressor(n_estimators=100, random_state=42),
    "GradientBoosting": GradientBoostingRegressor(n_estimators=100, random_state=42),
    "XGBoost": XGBRegressor(n_estimators=100, random_state=42),
    "LightGBM": LGBMRegressor(n_estimators=100, random_state=42),
    "NeuralNet": MLPRegressor(hidden_layer_sizes=(100, 50), max_iter=500, random_state=42)
}

# Dictionary to store results
results = {}

In [ ]:
# Train and evaluate each model
for name, model in models.items():
    score = cv_rmse(model)
    results[name] = score
    print(f"{name}: Mean RMSE = {score.mean():.4f}, Std = {score.std():.4f}")

# Create a table of RMSE scores
results_df = pd.DataFrame.from_dict(results, orient='index', columns=['Fold 1', 'Fold 2', 'Fold 3', 'Fold 4', 'Fold 5'])
results_df['Mean'] = results_df.mean(axis=1)
results_df['Std'] = results_df.std(axis=1)

print("\n--- Model Comparison (RMSE) ---")
print(results_df[['Mean', 'Std']].sort_values(by='Mean'))

# Visualize the model comparison
plt.figure(figsize=(10, 6))
sns.barplot(x=results_df.index, y=results_df['Mean'], order=results_df.sort_values(by='Mean').index)
plt.title('Model Comparison by Mean RMSE')
plt.xlabel('Model')
plt.ylabel('Mean RMSE (Log Scale)')
plt.xticks(rotation=45)
plt.show()

In [ ]:
# Select the best model (e.g., LightGBM)
best_model = LGBMRegressor(n_estimators=100, random_state=42)
best_model.fit(X, y)

# Get feature importances
importances = pd.DataFrame({
    'Feature': X.columns,
    'Importance': best_model.feature_importances_
}).sort_values(by='Importance', ascending=False).head(10)

# Visualize top 10 feature importances
plt.figure(figsize=(10, 6))
sns.barplot(x='Importance', y='Feature', data=importances)
plt.title('Top 10 Most Important Features')
plt.show()

In [ ]:
# Generate predictions on the test set
predictions = best_model.predict(X_test)

# Inverse the log transformation to get the actual sale prices
final_predictions = np.expm1(predictions)

# Create the submission file
submission = pd.DataFrame({
    'Id': test_ID,
    'SalePrice': final_predictions
})

# Save the submission file
submission.to_csv('submission.csv', index=False)

print("Submission file 'submission.csv' created successfully!")
print(submission.head())